### Phase 4: Backtesting-Strategie
Wir plotten zunächst die Regime-Wahrscheinlichkeiten der Modelle sowie der tatsächlichen Modell-Signale.

Anschließend testen wir den Erfolg einer Investition in Abhängigkeit zum gewählten Modell und den unten beschriebenen Regeln.

*   **Regel:**
    *   Wenn Modell sagt "Bull": 100% Aktien (S&P 500).
    *   Wenn Modell sagt "Bear": 100% Bonds oder Cash.
*   **Transaktionskosten:** Integriere realistische Kosten (z.B. 0,1% pro Trade), da ML-Modelle oft zu nervös hin- und herschalten ("Churning").

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
# --- 5. Dynamisches Backtesting (Automatisierter Vergleich aller Modelle) ---

import pandas as pd

sys.path.insert(0, "..")
from src.backtest.engine import run_all_backtests
from src.backtest.plots import plot_equity_curves

transaction_costs = cfg.transaction_cost_rate

backtesting_results, backtesting_transaction_costs = run_all_backtests(
    test_df=test_df,
    fee_rate=transaction_costs,
    signal_shift=cfg.backtesting.signal_shift,
)

equity_path = cfg.asset_path("equity_curves")
plot_equity_curves(backtesting_results, cfg.color_map, equity_path)

from IPython.display import Image, display
display(Image(filename=equity_path))

print(f"Backtesting abgeschlossen. {len([c for c in test_df.columns if c.endswith('_Signal')])} Strategien wurden berechnet.")
print("backtesting_results Dataframe")
print(backtesting_results)
print("backtesting_transaction_costs Dataframe")
print(backtesting_transaction_costs)

#--- Wir erhalten in diesem Schritt neben df und test_df backtesting_results mit kumulierten Werten ----
#--- Außerdem erhalten wir das Dataframe backtesting_transaction_costs mit Verläufen der Transaktionskosten ---

In [ ]:
# --- Performance & Drawdown Zusammenfassung ---

print("\n--- PERFORMANCE & DRAWDOWN ZUSAMMENFASSUNG ---")

# --- Performance-Summary aus src/ ---
from src.backtest.engine import calculate_performance_summary

performance_summary_df = calculate_performance_summary(backtesting_results)

# Als Markdown persistieren
performance_summary_df.to_markdown(cfg.asset_path("performance_summary"))

print("Zusammenfassung erfolgreich unter " + cfg.asset_path("performance_summary") + " gespeichert.")

# Im Notebook anzeigen
display(performance_summary_df)

In [ ]:
# Visualisierung der kumulierten Transaktionskosten
from src.backtest.plots import plot_transaction_costs

tc_path = cfg.asset_path("transaction_costs")
plot_transaction_costs(backtesting_transaction_costs, transaction_costs, cfg.color_map, tc_path)

from IPython.display import Image, display
display(Image(filename=tc_path))

In [ ]:
#--- SORR Simulation: Analyse der Entnahmephase ---

import matplotlib.pyplot as plt
import pandas as pd

from src.backtest.sorr import run_sorr_simulation, build_sorr_scenarios, build_sorr_summary
from src.backtest.plots import plot_sorr_scenario

scenarios = build_sorr_scenarios(cfg.backtesting.sorr.scenarios)
sorr_summaries = []
backtesting_sorr_simulation = pd.DataFrame(index=backtesting_results.index)

for name, params in scenarios.items():
    print(f"Running Scenario: {name}...")
    
    sim_results = run_sorr_simulation(
        backtesting_results, test_df, 
        params["start"], params["withdrawal"], params["fee"]
    )
    
    for col in sim_results.columns:
        backtesting_sorr_simulation[f"{name}_{col}"] = sim_results[col]
    
    sorr_path = cfg.asset_path(f"sorr_sim_{name.lower()}")
    plot_sorr_scenario(sim_results, name, params, cfg.color_map, sorr_path)
    
    from IPython.display import Image, display
    display(Image(filename=sorr_path))
    
    sorr_summaries.extend(build_sorr_summary(sim_results, name))

sorr_multi_eval = pd.DataFrame(sorr_summaries).set_index(['Szenario', 'Strategie'])
sorr_multi_eval.to_markdown(cfg.asset_path("sorr_summary"), index=True)

print("Alle Szenarien berechnet.")
display(sorr_multi_eval)
print(backtesting_sorr_simulation)

#--- Wir erhalten in diesem Schritt neben df, test_df, backtesting_results, backtesting_transaction_costs nun backtesting_sorr_simulation ----

In [ ]:
from pathlib import Path

output_path = Path(cfg.data_path('backtesting_results'))
output_path.parent.mkdir(parents=True, exist_ok=True)

backtesting_results.to_parquet(output_path)
print(f"Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_costs")

# Speichern als Parquet
backtesting_transaction_costs.to_parquet(output_path)

print(f"Transaktionskosten-Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_sorr")

# Speichern als Parquet
backtesting_sorr_simulation.to_parquet(output_path)

print(f"SORR-Simulations-Dataframe erfolgreich unter {output_path} gespeichert.")